In [1]:
import numpy as np
import torch

In [2]:


arr = np.load("results/activations/layer_18.npy", mmap_mode="r")
idx = [62004, 62005, 62006, 62048, 62049, 62050, 62092, 62093]  # distinct-text plateau members
rows = arr[idx].astype(np.float32)

for i in range(1, len(rows)):
    print(f"row0 vs row{i}: exact_equal={np.array_equal(rows[0], rows[i])}  max_abs_diff={np.abs(rows[0]-rows[i]).max():.6f}")


row0 vs row1: exact_equal=False  max_abs_diff=1.402832
row0 vs row2: exact_equal=False  max_abs_diff=1.172852
row0 vs row3: exact_equal=False  max_abs_diff=1.194336
row0 vs row4: exact_equal=False  max_abs_diff=1.387207
row0 vs row5: exact_equal=False  max_abs_diff=1.405273
row0 vs row6: exact_equal=False  max_abs_diff=1.480957
row0 vs row7: exact_equal=False  max_abs_diff=1.031586


In [5]:


sd = torch.load("results/trained_sae/L18_n15_k3_uncentered_s0/sae.pt", map_location="cpu")
w = sd["W_enc"][9].numpy()
b_dec = sd["b_dec"].numpy()

arr = np.load("results/activations/layer_18.npy", mmap_mode="r")
idx = [62004, 62005, 62006, 62048, 62049, 62050, 62092, 62093]
x = arr[idx].astype(np.float32) - b_dec

# dims driving the raw difference between row0 and row1
raw_diff = np.abs(x[0] - x[1])
top_raw = np.argsort(-raw_diff)[:10]
print("dims with the largest RAW activation difference (drive max_abs_diff):")
for d in top_raw:
    print(f"  dim {d:>5}: x0={x[0,d]:8.4f}  x1={x[1,d]:8.4f}  diff={raw_diff[d]:.4f}  W_enc[9][d]={w[d]:.5f}")

# dims W_enc[9] actually weights most heavily
top_w = np.argsort(-np.abs(w))[:10]
print("\ndims W_enc[9] weights most heavily (what the latent actually reads):")
for d in top_w:
    print(f"  dim {d:>5}: W_enc[9][d]={w[d]:8.4f}  x0={x[0,d]:.4f}  x1={x[1,d]:.4f}  diff={abs(x[0,d]-x[1,d]):.6f}")


dims with the largest RAW activation difference (drive max_abs_diff):
  dim  3266: x0=  0.2921  x1= -1.1108  diff=1.4028  W_enc[9][d]=0.01082
  dim  1162: x0= -0.6132  x1=  0.3250  diff=0.9382  W_enc[9][d]=0.28533
  dim   725: x0=  0.6316  x1= -0.0552  diff=0.6868  W_enc[9][d]=-0.18376
  dim  1039: x0= -0.2509  x1=  0.4115  diff=0.6624  W_enc[9][d]=-0.10827
  dim  3881: x0=  1.1662  x1=  0.5464  diff=0.6199  W_enc[9][d]=0.35663
  dim  4055: x0=  2.6789  x1=  2.0910  diff=0.5879  W_enc[9][d]=0.34945
  dim  2265: x0=  0.0409  x1= -0.5337  diff=0.5745  W_enc[9][d]=0.14248
  dim  2303: x0= -1.2663  x1= -0.7062  diff=0.5601  W_enc[9][d]=-0.39461
  dim  1731: x0= -0.4765  x1=  0.0484  diff=0.5249  W_enc[9][d]=0.05063
  dim  1565: x0= -0.7861  x1= -0.2630  diff=0.5231  W_enc[9][d]=0.32644

dims W_enc[9] weights most heavily (what the latent actually reads):
  dim  2742: W_enc[9][d]= -1.2312  x0=-1.8923  x1=-1.8904  diff=0.001953
  dim   357: W_enc[9][d]=  1.1403  x0=0.4041  x1=0.2375  diff=0.

In [8]:
sd = torch.load("results/trained_sae/L18_n15_k3_uncentered_s0/sae.pt", map_location="cpu")
W_enc9 = sd["W_enc"][9]
b_dec = sd["b_dec"]
b_enc9 = sd["b_enc"][9]

arr = np.load("results/activations/layer_18.npy", mmap_mode="r")
idx = [62004, 62005, 62006, 62048, 62049, 62050, 62092, 62093]
x = torch.from_numpy(arr[idx].astype(np.float32))

z_pre_9 = (x - b_dec) @ W_enc9 + b_enc9
print(z_pre_9)


tensor([6.4134, 5.4942, 6.1712, 5.0947, 4.6316, 5.0914, 4.4140, 5.2221])
